# Generic Silver MERGE helper
Attach lh_silver as default Lakehouse; use 3-part table names if supported in your workspace. Configure primary keys after discovering the source schema.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

def upsert_deduplicated(df, target, keys, updated_col=None):
    if not keys: raise ValueError('Business key required')
    if updated_col:
        w=Window.partitionBy(*keys).orderBy(F.col(updated_col).desc_nulls_last())
        df=df.withColumn('_rn',F.row_number().over(w)).filter('_rn=1').drop('_rn')
    else:
        # For nonidentical duplicates with same key, define a deterministic source ordering first.
        if df.groupBy(*keys).count().filter('count > 1').limit(1).count():
            raise ValueError('Duplicate business keys: supply deterministic updated_col')
    if not spark.catalog.tableExists(target):
        df.write.format('delta').mode('errorifexists').saveAsTable(target)
    else:
        condition=' AND '.join(f't.`{k}` = s.`{k}`' for k in keys)
        (DeltaTable.forName(spark,target).alias('t').merge(df.alias('s'),condition)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

# Example after mapping actual table and key:
# source = spark.table('lh_bronze.orders')
# upsert_deduplicated(source, 'orders', ['order_id'], 'updated_at')
